# Convert one DaVis result

Requires the source checkout and `python -m pip install -e ".[davis]"` from its
root. `lvpyio` is available on Windows/Linux, not macOS. This bundled file contains
pixel displacement: its component names and units must not be interpreted as
velocities. The recording date is an illustrative value, not recovered metadata.

In [ ]:
from datetime import datetime
from pathlib import Path
import h5py
from piv2hdf import PIVSnapshot, set_config, tutorial
from piv2hdf.davis import VC7File

source = tutorial.Davis.get_vc7_files()[0]
reader = VC7File(source)  # parameter information is read from this .vc7 file
snapshot = PIVSnapshot(reader, recording_dtime=datetime(2026, 1, 1, 12))

## Preserve the source calibration

The reader applies each component's slope and offset once and retains raw values.
Coordinate units come from the file. This example does not apply the planar PIV
convention or infer standard velocity names from vendor component names.

In [ ]:
with set_config(convention=None, postproc=[]):
    output = snapshot.to_hdf(
        hdf_filename=Path("outputs/davis_snapshot.hdf"),
        piv_attributes={"creator": {"name": "Example researcher"}},
    )
print(output)

In [ ]:
with h5py.File(output, "r") as data:
    print("x coordinate units:", data["x"].attrs["units"])
    print("U0:", data["U0"].shape, data["U0"].attrs["units"])
    print("source slope/offset:", data["U0"].attrs["source_slope"],
          data["U0"].attrs["source_offset"])
    print("raw component:", data["raw_U0"].shape)